# Synthetic Data Generator
### Step 11a : Build Synthetic Final Aligned Output Stage

This notebook builds the final Kafka/Postgres synthetic output so it matches the Kaggle-style dataset shape:

- `dataset_id`
- `run_id`
- `asset_id`
- `timestamp`
- `sensor_00` ... `sensor_51`
- `machine_status`

It uses the rebuilt stage as the source table and does **not** read premelt for this final output step.


## Overview

This notebook supports the synthetic pipeline stage documented by this notebook. It is part of the Synthetic support portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook supports the synthetic pipeline stage documented by this notebook.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify whether the notebook is a support, testing, streaming, or Bronze-handoff step in the synthetic path.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/synthetic_11a_build_final_aligned_observations_stage_code_reference.md`


In [1]:
# -----------------------------------------------------------------------------
# Stage 11A — Final Aligned Observations Stage
# -----------------------------------------------------------------------------
# Purpose:
# Build the full lineage-rich final aligned observation table.
#
# Source:
# - synthetic_observations_premelt_stage
# - synthetic_sensor_observations_rebuilt_stage
#
# Target:
# - synthetic_sensor_observations_final_aligned_stage

In [2]:
import os
import pandas as pd 

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
    execute_sql,
)

import pandas as pd

from utils.database.postgres import execute_sql, read_sql_dataframe

from utils.synthetic.pipeline.final_aligned_observation_writer import (
    build_final_aligned_observations_stage,
)

from utils.synthetic.pipeline.final_aligned_output_writer import build_synthetic_final_aligned_output_stage

from utils.synthetic.pipeline.final_aligned_incremental import run_final_align_loop

from utils.core.env_helpers import (
    env_bool, 
    env_int, 
    env_optional_int, 
    env_str,
)

pd.set_option("display.max_colwidth", None)



In [3]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)

IF_EXISTS_FLAG = "replace"

COMPLETE_ONLY_FLAG = env_bool(
    "SYNTHETIC_FINAL_ALIGN_COMPLETE_ONLY",
    True,
)

OBSERVATION_WINDOW_SIZE = env_int(
    "SYNTHETIC_REBUILD_OBSERVATION_WINDOW_SIZE",
    2500,
)

BATCH_SIZE = env_int(
    "SYNTHETIC_FINAL_ALIGN_BATCH_SIZE",
    1000,
)

MAX_ITERATIONS = env_optional_int(
    "SYNTHETIC_FINAL_ALIGN_MAX_ITERATIONS",
    default=None,
)

STOP_ON_FAILURE = env_bool(
    "SYNTHETIC_FINAL_ALIGN_STOP_ON_FAILURE",
    True,
)

PREMELT_TABLE_NAME = env_str(
    "SYNTHETIC_PREMELT_OBSERVATIONS_TABLE",
    "synthetic_observations_premelt_stage",
)

REBUILT_TABLE_NAME = env_str(
    "SYNTHETIC_REBUILT_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_rebuilt_stage",
)

TARGET_TABLE_NAME = env_str(
    "SYNTHETIC_FINAL_ALIGNED_OBSERVATIONS_TABLE",
    "synthetic_sensor_observations_final_aligned_stage",
    aliases=("FINAL_ALIGNMENT_SOURCE_TABLE_NAME",),
)

TIMESTAMP_SOURCE_PRIORITY = (
    "observation_timestamp",
    "timestamp",
    "created_at",
    "event_time",
)

STATUS_SOURCE_PRIORITY = (
    "machine_status",
    "stream_state",
    "phase",
)

STATUS_MAPPING = {
    "normal": "NORMAL",
    "broken": "BROKEN",
    "abnormal": "BROKEN",
    "failure": "BROKEN",
    "failed": "BROKEN",
    "fault": "BROKEN",
    "recovering": "RECOVERING",
    "recovery": "RECOVERING",
}

print("Final aligned config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("premelt table:", PREMELT_TABLE_NAME)
print("rebuilt table:", REBUILT_TABLE_NAME)
print("target table:", TARGET_TABLE_NAME)
print("batch size:", BATCH_SIZE)
print("max iterations:", MAX_ITERATIONS)

Final aligned config
schema: capstone
dataset id: pump_synthetic_v1
run id: synthetic_run_001
premelt table: synthetic_observations_premelt_stage
rebuilt table: synthetic_sensor_observations_rebuilt_stage
target table: synthetic_sensor_observations_final_aligned_stage
batch size: 1000
max iterations: None


---

In [4]:
engine = get_engine_from_env()

---

In [5]:
stage_11a_final_aligned_columns_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        ordinal_position,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = :schema_name
      AND table_name = :table_name
    ORDER BY ordinal_position
    """,
    params={
        "schema_name": SCHEMA,
        "table_name": TARGET_TABLE_NAME,
    },
)

display(stage_11a_final_aligned_columns_dataframe)

,ordinal_position,column_name,data_type
0,1,dataset_id,text
1,2,run_id,text
2,3,asset_id,text
3,4,generated_row_id,text
4,5,observation_index,bigint
...,...,...,...
66,67,sensor_51,double precision
67,68,rebuild_sensor_count,text
68,69,rebuild_is_complete,text
69,70,rebuild_completed_at,text


In [6]:
# DROP is triple-quoted to prevent accidental execution; Stage 11A
# only needs to drop the table when re-building from scratch after a schema change.
'''
execute_sql(
    engine,
    f"""
    DROP TABLE IF EXISTS "{SCHEMA}"."{TARGET_TABLE_NAME}" CASCADE
    """
)

print(f"Dropped stale Stage 11 target table: {SCHEMA}.{TARGET_TABLE_NAME}")
'''

'\nexecute_sql(\n    engine,\n    f"""\n    DROP TABLE IF EXISTS "{SCHEMA}"."{TARGET_TABLE_NAME}" CASCADE\n    """\n)\n\nprint(f"Dropped stale Stage 11 target table: {SCHEMA}.{TARGET_TABLE_NAME}")\n'

In [7]:
# -----------------------------------------------------------------------------
# Build full final aligned observation stage
# -----------------------------------------------------------------------------

stage_11a_result = build_final_aligned_observations_stage(
    engine=engine,
    schema=SCHEMA,
    premelt_table=PREMELT_TABLE_NAME,
    rebuilt_table=REBUILT_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=True,
    # Prefer rebuilt sensor values: they have been round-trip verified through
    # Kafka, whereas premelt values come directly from the generator.
    prefer_rebuilt_sensor_values=True,
    if_exists="replace",
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)

display(pd.DataFrame([stage_11a_result]))

[obs-window] 1 | observation_index 1 to 2,500
[final-align] Added 52 new columns to capstone.synthetic_sensor_observations_final_aligned_stage
[obs-window] 2 | observation_index 2,501 to 5,000
[obs-window] 3 | observation_index 5,001 to 7,500
[obs-window] 4 | observation_index 7,501 to 10,000
[obs-window] 5 | observation_index 10,001 to 12,500
[obs-window] 6 | observation_index 12,501 to 15,000
[obs-window] 7 | observation_index 15,001 to 17,500
[obs-window] 8 | observation_index 17,501 to 20,000
[obs-window] 9 | observation_index 20,001 to 22,500
[obs-window] 10 | observation_index 22,501 to 25,000
[obs-window] 11 | observation_index 25,001 to 27,500
[obs-window] 12 | observation_index 27,501 to 30,000
[obs-window] 13 | observation_index 30,001 to 32,500
[obs-window] 14 | observation_index 32,501 to 35,000
[obs-window] 15 | observation_index 35,001 to 37,500
[obs-window] 16 | observation_index 37,501 to 40,000
[obs-window] 17 | observation_index 40,001 to 42,500
[obs-window] 18 | obse

,status,premelt_rows,rebuilt_rows,final_rows,target_table
0,built,225000,225000,225000,synthetic_sensor_observations_final_aligned_stage


----

In [13]:
# -----------------------------------------------------------------------------
# Validate Stage 11A
# -----------------------------------------------------------------------------

stage_11a_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS final_row_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(DISTINCT generated_row_id) AS distinct_generated_row_id_count,
        COUNT(*) FILTER (WHERE observation_index IS NULL) AS null_observation_index_count,
        COUNT(*) FILTER (WHERE generated_row_id IS NULL) AS null_generated_row_id_count,
        COUNT(*) FILTER (WHERE observation_timestamp IS NULL) AS null_observation_timestamp_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS rebuild_complete_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp,
        (
            COUNT(*) = 225000
            AND COUNT(DISTINCT observation_index) = 225000
            AND COUNT(DISTINCT generated_row_id) = 225000
            AND COUNT(*) FILTER (WHERE observation_index IS NULL) = 0
            AND COUNT(*) FILTER (WHERE generated_row_id IS NULL) = 0
            AND COUNT(*) FILTER (WHERE observation_timestamp IS NULL) = 0
            AND COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) = 225000
            AND MIN(observation_index) = 1
            AND MAX(observation_index) = 225000
        ) AS ready_for_final_output_export
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_11a_validation_dataframe)

,final_row_count,distinct_observation_count,distinct_generated_row_id_count,null_observation_index_count,null_generated_row_id_count,null_observation_timestamp_count,rebuild_complete_count,min_observation_index,max_observation_index,min_observation_timestamp,max_observation_timestamp,ready_for_final_output_export
0,225000,225000,225000,0,0,225000,0,1,225000,None,None,False


---


### Sample Output

This shows the final Kaggle-shaped output table layout.


In [14]:
stage_11a_sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        generated_row_id,
        observation_index,
        observation_timestamp,
        stream_state,
        phase,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        rebuild_sensor_count,
        rebuild_is_complete
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    ORDER BY observation_index
    LIMIT 10
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_11a_sample_dataframe)

,dataset_id,run_id,asset_id,generated_row_id,observation_index,observation_timestamp,stream_state,phase,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,rebuild_sensor_count,rebuild_is_complete
0,pump_synthetic_v1,synthetic_run_001,pump_asset_001,synthetic_run_001_obs_000000000001,1,None,normal,normal,2.344329,48.327756,53.575950,43.424633,617.854335,None,None
1,pump_synthetic_v1,synthetic_run_001,pump_asset_001,synthetic_run_001_obs_000000000002,2,None,normal,normal,2.316564,49.037209,50.378763,47.447264,617.673971,None,None
2,pump_synthetic_v1,synthetic_run_001,pump_asset_001,synthetic_run_001_obs_000000000003,3,None,normal,normal,2.449135,48.920071,52.446083,46.391672,617.791244,None,None
3,pump_synthetic_v1,synthetic_run_001,pump_asset_001,synthetic_run_001_obs_000000000004,4,None,normal,normal,2.489110,49.304505,51.872589,42.288103,617.794481,None,None
4,pump_synthetic_v1,synthetic_run_001,pump_asset_001,synthetic_run_001_obs_000000000005,5,None,normal,normal,2.364396,45.769864,51.927495,41.465983,617.993983,None,None
5,pump_synthetic_v1,synthetic_run_001,pump_asset_001,synthetic_run_001_obs_000000000006,6,None,normal,normal,2.223095,47.166849,50.717235,40.841192,617.698302,None,None
6,pump_synthetic_v1,synthetic_run_001,pump_asset_001,synthetic_run_001_obs_000000000007,7,None,normal,normal,2.466576,48.560256,52.010692,44.414286,618.282081,None,None
7,pump_synthetic_v1,synthetic_run_001,pump_asset_001,synthetic_run_001_obs_000000000008,8,None,normal,normal,2.337132,43.087177,52.973015,44.378573,618.379612,None,None
8,pump_synthetic_v1,synthetic_run_001,pump_asset_001,synthetic_run_001_obs_000000000009,9,None,normal,normal,2.355965,45.514778,55.131609,43.304947,618.339956,None,None
9,pump_synthetic_v1,synthetic_run_001,pump_asset_001,synthetic_run_001_obs_000000000010,10,None,normal,normal,2.270712,46.130963,53.648135,42.564812,617.956074,None,None


---

In [15]:
stage_11a_status_distribution_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        stream_state,
        phase,
        COUNT(*) AS row_count
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    GROUP BY stream_state, phase
    ORDER BY stream_state, phase
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_11a_status_distribution_dataframe)

,stream_state,phase,row_count
0,broken,abnormal,7
1,normal,buildup,56339
2,normal,normal,154104
3,recovering,recovery,14550


---

In [16]:
# -----------------------------------------------------------------------------
# Validate Stage 11A
# -----------------------------------------------------------------------------

stage_11a_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS final_row_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(DISTINCT generated_row_id) AS distinct_generated_row_id_count,
        COUNT(*) FILTER (WHERE observation_index IS NULL) AS null_observation_index_count,
        COUNT(*) FILTER (WHERE generated_row_id IS NULL) AS null_generated_row_id_count,
        COUNT(*) FILTER (WHERE observation_timestamp IS NULL) AS null_observation_timestamp_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS rebuild_complete_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp,
        (
            COUNT(*) = 225000
            AND COUNT(DISTINCT observation_index) = 225000
            AND COUNT(DISTINCT generated_row_id) = 225000
            AND COUNT(*) FILTER (WHERE observation_index IS NULL) = 0
            AND COUNT(*) FILTER (WHERE generated_row_id IS NULL) = 0
            AND COUNT(*) FILTER (WHERE observation_timestamp IS NULL) = 0
            AND COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) = 225000
            AND MIN(observation_index) = 1
            AND MAX(observation_index) = 225000
        ) AS ready_for_final_output_export
    FROM "{SCHEMA}"."{TARGET_TABLE_NAME}"
    WHERE dataset_id = :dataset_id
      AND run_id = :run_id
    """,
    params={
        "dataset_id": DATASET_ID,
        "run_id": RUN_ID,
    },
)

display(stage_11a_validation_dataframe)

,final_row_count,distinct_observation_count,distinct_generated_row_id_count,null_observation_index_count,null_generated_row_id_count,null_observation_timestamp_count,rebuild_complete_count,min_observation_index,max_observation_index,min_observation_timestamp,max_observation_timestamp,ready_for_final_output_export
0,225000,225000,225000,0,0,225000,0,1,225000,None,None,False


---

In [17]:
# Dispose Engine
engine.dispose()

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Synthetic support step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

This support variant helps validate final-aligned output behavior for the synthetic path.
